# 📘 Notebook 0 — 第一次連線與移動

> 🎯 **目標**：連線到 Reachy Mini 並執行你的第一個動作指令。

---

## 0. 你將學到什麼

在本教學結束時，你將能夠：

* 了解 Reachy Mini 的架構（背景服務 daemon + 開發套件 SDK）
* 連線到你的機器人（實體硬體或模擬器）
* 執行基礎的移動指令
* 控制頭部與天線
* 練習安全地操作機器人

**預計時間：** 15-20 分鐘

> **開始之前：**  
> 請確保你在進入本教學前，已經完成 [`README.md`](README.md) 中環境需求區塊的所有設定步驟！

---

## 1. 架構總覽

Reachy Mini 使用的是 **主從式架構 (Client-Server)**：

```
┌─────────────────┐         ┌──────────────────┐
│  你的 Python 腳本 │         │   Reachy 背景服務│
│  / Notebook     │ ◄─────► │   (Daemon / 伺服端)│
│  (客戶端)       │         │                  │
└─────────────────┘         └────────┬─────────┘
                                     │
                                     ▼
                            ┌─────────────────┐
                            │  實體機器人硬體 │
                            │  或 模擬器      │
                            └─────────────────┘
```

**核心概念：**

* **Daemon (背景服務)**：直接安全地控制機器人馬達、感測器、相機與音訊的背景服務。
* **Python SDK**：你將用來傳送指令的 `reachy_mini` 套件。

**為什麼要用這種架構？**
- 允許多個客戶端同時連線（網頁 App、腳本、筆記本）。
- 背景服務能確保底層硬體的安全運作。
- 你可以透過網路遠端控制機器人。例如，你可以在高效能的伺服器上執行 AI 程式，而 Daemon 則跑在連接著機器人的 Raspberry Pi 上。

---

## 2. 確認連線狀態

在執行程式碼前，我們先確認機器人已經開機並正常運作。

你應該使用 **Reachy Mini Control** 來檢查機器人是否已連線並準備就緒。Reachy Mini Control 是一個桌面應用程式，讓你管理機器人、執行 App、播放表情並控制音效系統。如果你還沒安裝，請到 [官方網站](https://hf.co/reachy-mini/#/download) 下載。

在 Reachy Mini Control 連線後，確保機器人狀態為 **開啟 (ON)**。

**在繼續之前，請務必確認機器人是開啟的！特別是無線版本，機器人開機時預設是不會自動執行程式的。**

如果遇到問題，請參考文件中的 [連線與 Reachy Mini Control 疑難排解區塊](https://huggingface.co/docs/reachy_mini/troubleshooting#-connection--reachy-mini-control)。

---

## 3. 第一次連線

現在讓我們用 Reachy 的 Python SDK 來連線到機器人吧！

In [2]:
# 匯入 ReachyMini 類別
from reachy_mini import ReachyMini

# 連線到機器人
# 備註：此處可使用 media_backend="no_media" 先跳過相機/音訊
# 我們會在下一個筆記本中探索多媒體功能！
with ReachyMini() as mini:
    print("Successfully connected to Reachy Mini!")
    print(f"Robot name: {mini.robot_name}")

Successfully connected to Reachy Mini!
Robot name: reachy_mini


**重要筆記：**

* **`with` 語法**：這是 Python 的 Context Manager（上下文管理器），它可以確保你在執行完畢後自動、安全地關閉連線。

---

## 4. 🚨 安全性與最佳實踐

### 緊急停止

如果發生任何問題：
* **在 Jupyter 中**：點擊工具列的「停止」按鈕 (■)，或在終端機按下 `Ctrl+C`。
* **在 Python 腳本中**：按下 `Ctrl+C`。
* `with` 語法能確保程式強制退出時，機器人也會安全地停止動作。

### 最佳實踐

1. **總是使用 `with` 語法**
   ```python
   with ReachyMini() as mini:
       # 你的程式碼寫這
   # 連線會在這裡自動關閉
   ```

2. **在實驗階段先使用較長的時間 (duration)**（如 1-2 秒）
   - 較慢的動作比較安全。
   - 等你熟悉後再加快速度。

3. **先測試小幅度的動作**
   - 從小角度（5-10 度）開始測試。
   - 當你了解極限後再逐漸加大範圍。

4. **隨時注意機器人的狀況**
   - 觀察是否有非預期的動作。
   - 確保工作空間周圍淨空。

5. **在模擬器中嘗試**
   - 你也可以在模擬器中操作 Reachy Mini！
   - 只要在終端機啟動 Daemon 時加上 sim 參數：`reachy-mini-daemon --sim`。
   這會開啟一個 Mujoco 模擬視窗，讓你在沒有實體機的情況下看到並控制 Reachy Mini。這是安全測試程式碼的絕佳方法。關於模擬器的更多資訊，請參考文件中的 [模擬器區塊](https://huggingface.co/docs/reachy_mini/platforms/simulation/get_started)。

---

## 5. 第一次移動

讓 Reachy Mini 動起來吧！我們會先做出一個簡單的表情動作，然後再回到 **中立姿勢 (neutral position)** — 這是一個安全、置中的姿勢，也是後續教學中常會用到的基準點。

In [4]:
from reachy_mini.utils import create_head_pose

with ReachyMini() as mini:
    # 首先做出一個生動的姿勢：頭部傾斜，天線張開
    print("Moving to a curious pose...")
    mini.goto_target(
        head=create_head_pose(roll=20, degrees=True),  # 頭部向右傾斜
        antennas=[0.3, -0.3],  # 天線向外展開
        duration=2.0,
    )

    # 現在回到中立姿勢：頭部直視前方，天線直指上方
    print("Returning to neutral position...")
    mini.goto_target(
        head=create_head_pose(),  # 中立頭部姿勢（數值全為 0）
        antennas=[0.0, 0.0],  # 中立天線位置（弧度）
        duration=2.0,
    )

    print("Done!")

Moving to a curious pose...
Returning to neutral position...
Done!


**剛剛發生了什麼事？**

* **`goto_target()`**：讓機器人平滑地從現在的位置移動到目標位置。
* **`create_head_pose(roll=20, degrees=True)`**：建立一個向右傾斜 20 度的頭部姿勢。`degrees=True` 允許你使用角度而非弧度來設定。
* **`antennas=[0.3, -0.3]`**：讓兩根天線向外展開（右天線向內為 `+`，左天線向內為 `-`）。
* **`create_head_pose()`**：不加參數呼叫時，會建立 **中立姿勢**（x=0, y=0, z=0, roll=0, pitch=0, yaw=0），即頭部直視前方。
* **`antennas=[0.0, 0.0]`**：兩根天線直指上方 — 天線的中立姿勢。
* **`duration=2.0`**：這段動作會耗時 2 秒。時間越長 = 越慢越平滑；越短 = 越快。

> **提示：** 在兩個動作之間回到中立姿勢是個好習慣，這能給你一個可預期的起點並保持機器人的安全。

---

## 6. 移動頭部

頭部有 6 個自由度 (Degrees of freedom, DOF)：
- 平移 (Translation)：
     - X 軸
     - Y 軸
     - Z 軸
- 旋轉 (Rotation)：
     - 橫滾 (Roll)
     - 俯仰 (Pitch)
     - 偏航 (Yaw)

我們來逐一探索吧！

### 6.1. 平移 (Translation)

頭部可以沿著以下方向平移：
- 前後移動（沿著 X 軸）
- 左右移動（沿著 Y 軸）
- 上下移動（沿著 Z 軸）

<p align="center">
    <img src="https://github.com/pollen-robotics/reachy_mini/raw/main/docs/assets/reachy_mini_head_axis.png" alt="head axis" width ="40%"/>
</p>

In [6]:
with ReachyMini() as mini:
    # 從中立姿勢開始
    mini.goto_target(head=create_head_pose(), antennas=[0.0, 0.0], duration=1.0)

    # X 軸：前後平移
    print("Translate backwards...")
    mini.goto_target(
        head=create_head_pose(x=-0.02),  # 頭部向後平移 2 公分
        duration=1.0,
    )

    print("Translate forwards...")
    mini.goto_target(
        head=create_head_pose(x=0.02),  # 頭部向前平移 2 公分
        duration=1.0,
    )

    # 回到中立姿勢
    mini.goto_target(head=create_head_pose(), duration=1.0)

    # Y 軸：左右平移
    print("Translate to the right...")
    mini.goto_target(
        head=create_head_pose(y=-0.02),  # 頭部向右平移 2 公分
        duration=1.0,
    )

    print("Translate to the left...")
    mini.goto_target(
        head=create_head_pose(y=0.02),  # 頭部向左平移 2 公分
        duration=1.0,
    )

    # 回到中立姿勢
    mini.goto_target(head=create_head_pose(), duration=1.0)

    # Z 軸：上下平移
    print("Translate downwards...")
    mini.goto_target(
        head=create_head_pose(z=-0.02),  # 頭部向下平移 2 公分
        duration=1.0,
    )

    print("Translate upwards...")
    mini.goto_target(
        head=create_head_pose(z=0.02),  # 頭部向上平移 2 公分
        duration=1.0,
    )

    # 回到中立姿勢
    mini.goto_target(head=create_head_pose(), duration=1.0)

    print("Done!")

Translate backwards...
Translate forwards...
Translate to the right...
Translate to the left...
Translate downwards...
Translate upwards...
Done!


**剛剛發生了什麼事？**

* **`goto_target()`**：讓機器人平滑地從當前位置移動到目標位置。
* **`create_head_pose()`**：根據你給的參數建立頭部姿勢。在這裡我們只修改了平移的數值 (x, y 或 z)。當然，你也可以同時修改所有參數。

### 6.2. 旋轉 (Rotation)

頭部可以沿著三個軸旋轉：

* **Roll (橫滾)**：左右傾斜（繞 X 軸旋轉）
  - 正值 = 向右傾斜
  - 負值 = 向左傾斜

* **Pitch (俯仰)**：點頭（繞 Y 軸旋轉）
  - 正值 = 低頭看
  - 負值 = 抬頭看

* **Yaw (偏航)**：搖頭（繞 Z 軸旋轉）
  - 正值 = 向左看
  - 負值 = 向右看

<p align="center">
    <img src="https://github.com/pollen-robotics/reachy_mini/raw/main/docs/assets/reachy_mini_head_rotation.png" alt="head rotation" width ="40%"/>
</p>

In [7]:
with ReachyMini() as mini:
    # 從中立姿勢開始
    mini.goto_target(head=create_head_pose(), antennas=[0.0, 0.0], duration=1.0)

    # Roll (橫滾)：傾斜頭部（如困惑的表情，向右與向左）
    print("Tilting head right...")
    mini.goto_target(head=create_head_pose(roll=20, degrees=True), duration=1.0)

    # 回到中立姿勢
    mini.goto_target(head=create_head_pose(), duration=1.0)

    print("Tilting head left...")
    mini.goto_target(head=create_head_pose(roll=-20, degrees=True), duration=1.0)

    # 回到中立姿勢
    mini.goto_target(head=create_head_pose(), duration=1.0)

    # Pitch (俯仰)：點頭「是」（向下與向上）
    print("Nodding head down...")
    mini.goto_target(head=create_head_pose(pitch=15, degrees=True), duration=1.0)

    print("Nodding head up...")
    mini.goto_target(head=create_head_pose(pitch=-15, degrees=True), duration=1.0)

    # 回到中立姿勢
    mini.goto_target(head=create_head_pose(), duration=1.0)

    # Yaw (偏航)：搖頭「否」（向左與向右）
    print("Shaking head left...")
    mini.goto_target(head=create_head_pose(yaw=30, degrees=True), duration=1.0)

    print("Shaking head right...")
    mini.goto_target(head=create_head_pose(yaw=-30, degrees=True), duration=1.0)

    # 回到中立姿勢
    mini.goto_target(head=create_head_pose(), duration=1.0)

    print("Done!")

Tilting head right...
Tilting head left...
Nodding head down...
Nodding head up...
Shaking head left...
Shaking head right...
Done!


**重要：** 使用角度時，請務必加上 `degrees=True` 參數！

Reachy Mini 具有硬體與軟體上的限制，以防止自我碰撞與損壞。SDK 會自動將數值限制在最接近的安全位置。舉例來說，Roll 的極限是 +/- 40 度，如果你試圖設定為 50 度，它會被強制限制在 40 度。
完整的可動範圍與限制，請參考文件中的 [核心概念區塊 (core concepts)](https://huggingface.co/docs/reachy_mini/SDK/core-concept#safety-limits-)。

---

## 7. 移動天線

天線是表達情緒與展現注意力的絕佳工具！

In [8]:
import numpy as np

with ReachyMini() as mini:
    # 從中立姿勢開始
    mini.goto_target(head=create_head_pose(), antennas=[0.0, 0.0], duration=1.0)

    # 兩根天線向外擺動（興奮/警覺）
    print("Excited!")
    for _ in range(5):
        mini.goto_target(
            head=create_head_pose(),
            antennas=[
                np.deg2rad(-10),
                np.deg2rad(10),
            ],  # 右天線向外 (-)，左天線向外 (+)
            duration=0.1,
        )
        mini.goto_target(
            head=create_head_pose(),
            antennas=[
                np.deg2rad(10),
                np.deg2rad(-10),
            ],  # 右天線向內 (+)，左天線向內 (-)
            duration=0.1,
        )

    # 兩根天線向下垂（悲傷/疲倦）
    print("Sad...")
    mini.goto_target(
        head=create_head_pose(),
        antennas=[
            np.deg2rad(-140),
            np.deg2rad(140),
        ],  # Right outward (-), Left outward (+)
        duration=3.0,
    )

    # 回到中立姿勢
    mini.goto_target(head=create_head_pose(), antennas=[0.0, 0.0], duration=1.0)

    # 交替不對稱（思考/困惑）
    print("Confused?")
    mini.goto_target(
        head=create_head_pose(),
        antennas=[np.deg2rad(-60), np.deg2rad(-30)],
        duration=1.0,
    )  # 右天線向外，左天線向內 - 設定不同數值營造困惑效果

    # 回到中立姿勢
    mini.goto_target(head=create_head_pose(), antennas=[0.0, 0.0], duration=1.0)

    print("Done!")

Excited!
Sad...
Confused?
Done!


**天線控制提示：**

* 天線的設定格式為 `[右天線, 左天線]`，單位為 **弧度 (radians)** 且方向是一致的：**正值**的指令會讓天線向**右**移動，**負值**的指令會讓天線向**左**移動。因此，如果你想要做出對稱的動作，兩邊的指令數值必須相反：
  - 右天線：
    - 負值向外傾斜
    - 正值向內傾斜
  - 左天線：
    - 正值向外傾斜
    - 負值向內傾斜
* 你可以將角度轉換為弧度：`np.deg2rad(45)` 或 `np.radians(45)`
* 天線非常適合用來展現個性！別猶豫，多嘗試不同的天線動作來賦予 Reachy Mini 更多角色性格。

---

## 8. 組合頭部與天線的動作

讓我們把頭部和天線結合起來，創造更複雜的表情吧！

In [9]:
with ReachyMini() as mini:
    # 好奇的表情：頭部傾斜 + 不對稱天線
    print("Curious...")
    mini.goto_target(
        head=create_head_pose(roll=20, degrees=True),
        antennas=[np.deg2rad(-60), np.deg2rad(-30)],
        duration=2.0,
    )

    # 回到中立姿勢
    mini.goto_target(head=create_head_pose(), antennas=[0.0, 0.0], duration=1.0)

    # 悲傷：低頭 + 天線下垂
    print("Sad...")
    mini.goto_target(
        head=create_head_pose(pitch=30, degrees=True),
        antennas=(np.deg2rad(-140), np.deg2rad(140)),
        duration=3.0,
    )

    # 回到中立姿勢
    mini.goto_target(head=create_head_pose(), antennas=[0.0, 0.0], duration=1.0)

    print("Done!")

Curious...
Sad...
Done!


**組合動作提示：**

**你可以創造出極具表現力的行為**，只要將頭部方向與天線位置同步即可。例如：
- 悲傷的表情：低頭 + 天線向外垂下
- 好奇的表情：頭部傾斜 + 不對稱的天線角度
- 興奮的表情：抬頭 + 天線快速擺動
* **時間點是關鍵**：對於情緒表達，使用較長的持續時間（2-3 秒）能讓動作看起來更可信且自然。
* **嘗試不對稱**：不同的天線位置能創造出更多的個性和特點。

---

## 9. 播放預錄的情緒

雖然手動設計每一個姿勢很有表現力，但 Reachy Mini 也內建了一個 **預錄情緒資料庫** — 這些是結合了頭部與天線的完整動作序列，用來表達快樂、驚訝、無聊等情感。

這些資料儲存在 HuggingFace 的 [pollen-robotics/reachy-mini-emotions-library](https://huggingface.co/datasets/pollen-robotics/reachy-mini-emotions-library) 資料集中，並可直接透過 SDK 串流與播放。你可以在 [Reachy Mini 情緒展示 App (Emotions App)](https://huggingface.co/spaces/RemiFabre/emotions) 中預覽所有的情緒。

讓我們載入資料庫並播放幾個看看吧！

In [10]:
from reachy_mini.motion.recorded_move import RecordedMoves

EMOTIONS_DATASET = "pollen-robotics/reachy-mini-emotions-library"

emotions = RecordedMoves(EMOTIONS_DATASET)

print(f"{len(emotions.list_moves())} emotions available:")
print(emotions.list_moves())

85 emotions available:
['amazed1', 'anxiety1', 'attentive1', 'attentive2', 'boredom1', 'boredom2', 'calming1', 'cheerful1', 'come1', 'confused1', 'contempt1', 'curious1', 'dance1', 'dance2', 'dance3', 'disgusted1', 'displeased1', 'displeased2', 'downcast1', 'dying1', 'electric1', 'enthusiastic1', 'enthusiastic2', 'exhausted1', 'fear1', 'frustrated1', 'furious1', 'go_away1', 'grateful1', 'helpful1', 'helpful2', 'impatient1', 'impatient2', 'incomprehensible2', 'indifferent1', 'inquiring1', 'inquiring2', 'inquiring3', 'irritated1', 'irritated2', 'laughing1', 'laughing2', 'lonely1', 'lost1', 'loving1', 'mini-deep-sleep', 'no1', 'no_excited1', 'no_sad1', 'oops1', 'oops2', 'proud1', 'proud2', 'proud3', 'rage1', 'relief1', 'relief2', 'reprimand1', 'reprimand2', 'reprimand3', 'resigned1', 'sad1', 'sad2', 'scared1', 'serenity1', 'shy1', 'sleep1', 'success1', 'success2', 'surprised1', 'surprised2', 'thoughtful1', 'thoughtful2', 'tired1', 'toc-toc-toc', 'uncertain1', 'uncomfortable1', 'understand

這個應用程式中共有 81 種情緒可供使用，你也可以透過 SDK 控制頭部與天線來創造專屬於你的客製化情緒。這是一個練習並發揮 Reachy Mini 表現力創意的好方法！

讓我們從 App 中隨機挑選一個情緒來播放，看看在機器人上會是什麼樣子！你可以多次執行這個區塊來觀看不同的情緒。每次執行時，它都會從 81 種情緒中隨機挑選一個，並在你的 Reachy Mini 上執行對應的頭部與天線動作。這是探索機器人豐富表情的有趣方式！

你可以試著在你的機器人上重現它們，並以此作為你自己設計表情的靈感！

In [11]:
# 從資料集中隨機挑選一個情緒並播放
import random

with ReachyMini() as mini:
    emotion_name = random.choice(emotions.list_moves())
    print(f"Playing '{emotion_name}'...")
    await mini.async_play_move(emotions.get(emotion_name), initial_goto_duration=1.0)
    print("Done!")

Playing 'success1'...
Done!


## 10. 練習（自己試試看！）

### 練習 1：設計一個打招呼的動作

建立一個連續動作：
1. 微微抬頭看
2. 將頭偏向一側
3. 擺動天線
4. 回到中立姿勢

**提示：** 連續使用多次 `goto_target()` 來達成。

<details>
<summary><b>💡 點擊顯示解答</b></summary>

```python
from reachy_mini import ReachyMini
from reachy_mini.utils import create_head_pose

with ReachyMini(media_backend="no_media") as mini:
    # 1. 微微抬頭
    mini.goto_target(
        head=create_head_pose(pitch=-10, degrees=True),
        antennas=[0.0, 0.0],
        duration=1.0
    )
    
    # 2. 將頭偏向一側
    mini.goto_target(
        head=create_head_pose(roll=15, pitch=-10, degrees=True),
        duration=1.0
    )
    
    # 3. 擺動天線 - 還記得天線控制部分的興奮動作嗎？
    for _ in range(5):
        mini.goto_target(
        antennas=[-0.3, 0.3],
        duration=0.1
        )
        mini.goto_target(
            antennas=[0.3, -0.3],
            duration=0.1
        )
    
    # 4. 回到中立姿勢
    mini.goto_target(
        head=create_head_pose(),
        antennas=[0.0, 0.0],
        duration=1.5
    )
    
    print("打招呼完成！")
```

</details>

In [ ]:
# 在此撰寫你的程式碼
from reachy_mini import ReachyMini
from reachy_mini.utils import create_head_pose

with ReachyMini() as mini:
    # TODO: 設計你的打招呼動作
    pass

### 練習 2：實驗不同的持續時間 (Duration)

將頭部從中立姿勢移動到 `yaw=30°`，重複三次並使用不同的時間：
- 0.5 秒（快）
- 2.0 秒（正常）
- 5.0 秒（慢）

每次移動後請回到中立姿勢。觀察動作速度的變化。

<details>
<summary><b>💡 點擊顯示解答</b></summary>

```python
from reachy_mini import ReachyMini
from reachy_mini.utils import create_head_pose

with ReachyMini(media_backend="no_media") as mini:
    # 從中立姿勢開始
    mini.goto_target(head=create_head_pose(), duration=1.0)
    
    # 快速移動 (0.5s)
    print("快速移動 (0.5s)...")
    mini.goto_target(
        head=create_head_pose(yaw=30, degrees=True),
        duration=0.5
    )
    mini.goto_target(head=create_head_pose(), duration=0.5)
    
    # 正常移動 (2.0s)
    print("正常移動 (2.0s)...")
    mini.goto_target(
        head=create_head_pose(yaw=30, degrees=True),
        duration=2.0
    )
    mini.goto_target(head=create_head_pose(), duration=2.0)
    
    # 緩慢移動 (5.0s)
    print("緩慢移動 (5.0s)...")
    mini.goto_target(
        head=create_head_pose(yaw=30, degrees=True),
        duration=5.0
    )
    mini.goto_target(head=create_head_pose(), duration=5.0)
    
    print("時間實驗完成！")
```

</details>

In [ ]:
# 在此撰寫你的程式碼
from reachy_mini import ReachyMini
from reachy_mini.utils import create_head_pose

with ReachyMini() as mini:
    # TODO: 測試不同的持續時間 (duration)
    pass

### 練習 3：設計一個「專注」動畫

讓 Reachy 看起來像是在專注聆聽：
1. 從中立開始
2. 稍微看向左邊
3. 稍微看向右邊
4. 轉回前方並稍微傾斜頭部（聆聽姿勢）
5. 回到中立姿勢

試著讓動作流暢自然！

<details>
<summary><b>💡 點擊顯示解答</b></summary>

```python
from reachy_mini import ReachyMini
from reachy_mini.utils import create_head_pose

with ReachyMini(media_backend="no_media") as mini:
    # 1. 從中立開始
    print("開始專注動畫...")
    mini.goto_target(
        head=create_head_pose(),
        antennas=[0.0, 0.0],
        duration=1.0
    )
    
    # 2. 稍微看向左邊
    mini.goto_target(
        head=create_head_pose(yaw=25, degrees=True),
        duration=1.5
    )
    
    # 3. 稍微看向右邊
    mini.goto_target(
        head=create_head_pose(yaw=-25, degrees=True),
        duration=2.0
    )
    
    # 4. 轉回前方並稍微傾斜頭部（聆聽姿勢）
    mini.goto_target(
        head=create_head_pose(roll=20, degrees=True),
        antennas=[-1.0, -0.5],
        duration=2.0
    )
    
    # 5. 回到中立姿勢
    mini.goto_target(
        head=create_head_pose(),
        antennas=[0.0, 0.0],
        duration=2.0
    )
    
    print("專注動畫完成！")
```

**提示：** 看起來自然的關鍵在於使用適合的時間，以及姿勢之間平滑的過渡。試著實驗不同的時間設定，找出看起來最棒的效果！

</details>

In [ ]:
# 在此撰寫你的程式碼
from reachy_mini import ReachyMini
from reachy_mini.utils import create_head_pose

with ReachyMini() as mini:
    # TODO: 設計專注聆聽動畫
    pass

---

## 10. 下一步？

恭喜你！你已經學會如何：
* ✅ 連線到 Reachy Mini
* ✅ 使用 pitch、yaw 和 roll 來移動頭部
* ✅ 控制天線
* ✅ 創造組合動作
* ✅ 遵循安全最佳實踐

在下一個筆記本中，你將學到：
* 📸 如何從相機獲取影像
* 🎤 如何透過麥克風錄製聲音
* 🔊 如何透過揚聲器播放音效

➡️ **下一步：Notebook 1 — 基礎多媒體（相機與音訊）** 📷🎵